In [1]:
import os, pandas as pd

In [2]:
base = "/kaggle/input/datasets/ozguraslank/brain-stroke-ct-dataset/Brain_Stroke_CT_Dataset"
rows = []
for cls in ["Normal", "Bleeding", "Ischemia"]:
    for f in sorted(os.listdir(f"{base}/{cls}/PNG")):
        rows.append({"id": f[:-4], "cls": cls,
                     "png": f"{base}/{cls}/PNG/{f}",
                     "dcm": f"{base}/{cls}/DICOM/{f[:-4]}.dcm"})
df = pd.DataFrame(rows)
df["bleed"] = (df["cls"] == "Bleeding").astype(int)
print(df["cls"].value_counts())
print("ids unique:", df["id"].is_unique)
print("bleed share:", round(df["bleed"].mean(), 3))
print("always-no-bleed baseline accuracy:", round(1 - df["bleed"].mean(), 3), "| bleeds caught: 0")

cls
Normal      4427
Ischemia    1130
Bleeding    1093
Name: count, dtype: int64
ids unique: True
bleed share: 0.164
always-no-bleed baseline accuracy: 0.836 | bleeds caught: 0


In [3]:
# !pip install -q pydicom

In [4]:
import pydicom

In [5]:
d = pydicom.dcmread(df["dcm"].iloc[0], stop_before_pixels=True)
for t in ["PatientID","StudyInstanceUID","SeriesInstanceUID","InstanceNumber","SliceLocation","PatientAge","PatientSex","Modality","Rows","Columns"]:
    print(t, "->", getattr(d, t, "MISSING"))

sample = df.sample(300, random_state=0)
def tag(p, t):
    try: return str(getattr(pydicom.dcmread(p, stop_before_pixels=True), t, "MISSING"))
    except Exception: return "ERR"
for t in ["PatientID", "StudyInstanceUID", "SeriesInstanceUID"]:
    print(t, "unique values in 300 random files:", sample["dcm"].map(lambda p: tag(p, t)).nunique())

PatientID -> MISSING
StudyInstanceUID -> MISSING
SeriesInstanceUID -> MISSING
InstanceNumber -> MISSING
SliceLocation -> MISSING
PatientAge -> MISSING
PatientSex -> MISSING
Modality -> MISSING
Rows -> 512
Columns -> 512
PatientID unique values in 300 random files: 1
StudyInstanceUID unique values in 300 random files: 1
SeriesInstanceUID unique values in 300 random files: 1


In [6]:
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [7]:
SEED = 42
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["cls"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["cls"], random_state=SEED)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), d["cls"].value_counts().to_dict())

train 4655 {'Normal': 3099, 'Ischemia': 791, 'Bleeding': 765}
val 997 {'Normal': 664, 'Ischemia': 169, 'Bleeding': 164}
test 998 {'Normal': 664, 'Ischemia': 170, 'Bleeding': 164}


In [8]:
def report(name, y_true, y_pred, cls_true):
    y_true, y_pred, cls_true = np.asarray(y_true), np.asarray(y_pred), np.asarray(cls_true)
    tp = int(((y_true == 1) & (y_pred == 1)).sum()); fn = int(((y_true == 1) & (y_pred == 0)).sum())
    print(name)
    print(f"  accuracy     {np.mean(y_true == y_pred):.3f}")
    print(f"  bleed recall {tp / (tp + fn):.3f}   (missed bleeds: {fn} of {tp + fn})")
    for c in ["Normal", "Ischemia"]:
        m = cls_true == c
        print(f"  false alarms on {c}: {int((y_pred[m] == 1).sum())} of {int(m.sum())}")

def load_small(paths, size=32):
    return np.stack([np.asarray(Image.open(p).convert("L").resize((size, size)), dtype=np.float32).ravel() / 255.0
                     for p in paths])

report("Baseline: always 'no bleed' (validation)", val_df["bleed"], np.zeros(len(val_df), dtype=int), val_df["cls"])

Xtr, Xva = load_small(train_df["png"]), load_small(val_df["png"])
lr = LogisticRegression(max_iter=300, class_weight="balanced")
lr.fit(Xtr, train_df["bleed"])
report("Baseline 2: logistic regression on 32x32 pixels (validation)", val_df["bleed"], lr.predict(Xva), val_df["cls"])

Baseline: always 'no bleed' (validation)
  accuracy     0.836
  bleed recall 0.000   (missed bleeds: 164 of 164)
  false alarms on Normal: 0 of 664
  false alarms on Ischemia: 0 of 169
Baseline 2: logistic regression on 32x32 pixels (validation)
  accuracy     0.825
  bleed recall 0.744   (missed bleeds: 42 of 164)
  false alarms on Normal: 103 of 664
  false alarms on Ischemia: 29 of 169


In [9]:
import torch, torch.nn as nn, torchvision, numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score

In [10]:
torch.manual_seed(42)
CLASSES = ["Normal", "Bleeding", "Ischemia"]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [11]:
tf = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

In [13]:
class CT(Dataset):
    def __init__(self, d):
        self.p = d["png"].tolist()
        self.y = [CLASSES.index(c) for c in d["cls"]]
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

train_dl = DataLoader(CT(train_df), batch_size=64, shuffle=True, num_workers=2)
val_dl = DataLoader(CT(val_df), batch_size=64, shuffle=False, num_workers=2)

In [14]:
model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(512, 3)
model = model.to(device)

counts = train_df["cls"].value_counts()
w = torch.tensor([len(train_df) / (3 * counts[c]) for c in CLASSES], dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

def predict(dl):
    model.eval(); out = []
    with torch.no_grad():
        for xb, _ in dl:
            out.append(model(xb.to(device)).softmax(1).cpu().numpy())
    return np.concatenate(out)

best_auc, best_state = 0, None
for epoch in range(5):
    model.train()
    for xb, yb in train_dl:
        opt.zero_grad()
        loss = loss_fn(model(xb.to(device)), yb.to(device))
        loss.backward(); opt.step()
    p_bleed = predict(val_dl)[:, 1]
    auc = roc_auc_score(val_df["bleed"], p_bleed)
    print(f"epoch {epoch + 1}: validation bleed AUC {auc:.3f}")
    if auc > best_auc:
        best_auc, best_state = auc, {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
p_bleed = predict(val_dl)[:, 1]
report("ResNet18 fine-tuned (validation)", val_df["bleed"], (p_bleed >= 0.5).astype(int), val_df["cls"])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]


epoch 1: validation bleed AUC 0.983
epoch 2: validation bleed AUC 0.990
epoch 3: validation bleed AUC 0.994
epoch 4: validation bleed AUC 0.993
epoch 5: validation bleed AUC 0.988
ResNet18 fine-tuned (validation)
  accuracy     0.972
  bleed recall 0.927   (missed bleeds: 12 of 164)
  false alarms on Normal: 11 of 664
  false alarms on Ischemia: 5 of 169


In [15]:
ext = pd.read_csv(f"{base}/External_Test/labels.csv")
print("External_Test ids also in main data:", ext["image_id"].astype(str).isin(df["id"]).sum(), "of", len(ext))

External_Test ids also in main data: 0 of 200


In [16]:
import torch, numpy as np, pandas as pd

In [17]:
torch.save(best_state, "/kaggle/working/resnet18_ct_best.pt")

In [18]:
def probs_for(paths):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(paths), 64):
            xb = torch.stack([tf(Image.open(p).convert("RGB")) for p in paths[i:i + 64]]).to(device)
            out.append(model(xb).softmax(1).cpu().numpy())
    return np.concatenate(out)

def summarise(name, y, pred, pos_name):
    y, pred = np.asarray(y), np.asarray(pred)
    tp = int(((y == 1) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum()); neg = int((y == 0).sum())
    print(f"{name}")
    print(f"  {pos_name} recall {tp / (tp + fn):.3f}  (missed {fn} of {tp + fn})")
    print(f"  false alarms {fp} of {neg}")

# 1) Same-source held-out set (External_Test). Label is "any stroke", so use 1 - P(Normal)
ext = pd.read_csv(f"{base}/External_Test/labels.csv")
P_ext = probs_for([f"{base}/External_Test/PNG/{i}.png" for i in ext["image_id"]])
summarise("External_Test (same source, any stroke)", ext["Stroke"].values, (1 - P_ext[:, 0] >= 0.5).astype(int), "stroke")

# 2) Different source: Kitamura Head CT (normal vs hemorrhage), use P(Bleeding)
kb = "/kaggle/input/datasets/felipekitamura/head-ct-hemorrhage"
kl = pd.read_csv(f"{kb}/labels.csv"); kl.columns = [c.strip() for c in kl.columns]
lab = kl.columns[1]
print(kl.head(3)); print(kl[lab].value_counts().to_dict())
def kpath(i):
    s = str(i)
    return f"{kb}/head_ct/head_ct/{s}" if s.endswith(".png") else f"{kb}/head_ct/head_ct/{int(i):03d}.png"
P_k = probs_for([kpath(i) for i in kl.iloc[:, 0]])
summarise("Kitamura Head CT (different source, bleed vs normal)", kl[lab].values, (P_k[:, 1] >= 0.5).astype(int), "bleed")

External_Test (same source, any stroke)
  stroke recall 0.843  (missed 11 of 70)
  false alarms 12 of 130
   id  hemorrhage
0   0           1
1   1           1
2   2           1
{1: 100, 0: 100}
Kitamura Head CT (different source, bleed vs normal)
  bleed recall 1.000  (missed 0 of 100)
  false alarms 59 of 100
